## Setup

In [ ]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.status()

In [ ]:
using CairoMakie
using Carlo.ResultTools
using CarloAnalysis
using DataFrames
using FFTW
using HDF5
using JLD2
using LinearAlgebra
using StaticArrays

set_theme!(theme_latexfonts())

In [ ]:
# Boltzmann constant in meV/K
const kB = 8.617333262e-2

In [ ]:
getetapar(ηk) = real(ηk[1,1] + ηk[2,2])
getetapar(ηk::Vector) = getetapar(reshape(ηk, (3,3)))

In [ ]:
function generate_spins(jobname, task_no)
    fig = Figure(size=(800, 400))

    task_str = lpad(task_no, 4, "0")
    h5open("../jobs/$jobname.data/task$task_str/run0001.dump.h5") do file
        spins = map(
            t -> [t[:data][1], t[:data][2], t[:data][3]],
            read(file, "simulation/spins")
        )
        spin_xs = map(v -> v[1], spins)
        spin_ys = map(v -> v[2], spins)
        spin_zs = map(v -> v[3], spins)
        Lx, Ly = size(spins)
        fig[1,1] = Axis(fig; title="Spins", backgroundcolor="black")
        strength = vec(spin_zs)
        arrows2d!(1:Lx, 1:Ly, spin_xs, spin_ys, lengthscale=0.5, align=:center, color=strength,
                  colorrange=(-1, 1))

        ηs = map(
            t -> [t[:data][1], t[:data][2], t[:data][3]],
            read(file, "simulation/etas")
        )
        η_xs = getindex.(ηs, 1)
        η_ys = getindex.(ηs, 2)
        η_zs = getindex.(ηs, 3)
        Lx, Ly = size(ηs)
        fig[1,2] = Axis(fig; title="ηs", backgroundcolor="black")
        strength = vec(η_zs)
        arrows2d!(1:Lx, 1:Ly, η_xs, η_ys, lengthscale=0.5, align=:center, color=strength,
                  colorrange=(-1, 1))
    end

    return fig
end

In [ ]:
function generate_spinks(jobname, task_no; run_no=1)
    fig = Figure(size=(800, 500))

    task_str = lpad(task_no, 4, "0")
    run_str = lpad(run_no, 4, "0")
    h5open("../jobs/$jobname.data/task$task_str/run$run_str.dump.h5") do file
        rawetas = map(
            t -> [t[:data][1], t[:data][2], t[:data][3]],
            read(file, "simulation/etas")
        )
        etas = zeros(size(rawetas)..., 3)
        for I in eachindex(IndexCartesian(), rawetas)
            etas[I, :] = rawetas[I]
        end
        etaks = fft(etas, (1, 2)) / length(rawetas)
        spin_mags = sum(abs2.(etaks), dims=3)[:,:,1]
        fig[1,1] = ax = Axis(fig; title="ηk correlations")
        hm = heatmap!(ax, spin_mags)
        Colorbar(fig[2,1], hm, vertical=false, flipaxis=false)

        rawspins = map(
            t -> [t[:data][1], t[:data][2], t[:data][3]],
            read(file, "simulation/spins")
        )
        spins = zeros(size(rawspins)..., 3)
        for I in eachindex(IndexCartesian(), rawspins)
            spins[I, :] = rawspins[I]
        end
        spinks = fft(spins, (1, 2)) / length(rawspins)
        spin_mags = sum(abs2.(spinks), dims=3)[:,:,1]
        fig[1,2] = ax = Axis(fig; title="sk correlations")
        hm = heatmap!(ax, spin_mags)
        Colorbar(fig[2,2], hm, vertical=false, flipaxis=false)
    end

    return fig
end

## am = 4

In [ ]:
results = JobResult("../jobs", "am4")

In [ ]:
CairoMakie.activate!()

fig = Figure(size=(800, 400))
df = subset(results.data, :init_type => x -> x .== "stripe")
ax1 = fig[1,1] = Axis(fig,
    title="S correlation half M vs. T (stripe init)", xlabel=L"T",
    ylabel=L"C_{M/2}"
)
generate_plot!(ax1, :T, :sk_corr_half_M, [:er], df; line=true)
fig[1,2] = ax2 = Axis(fig,
    title="η correlation M vs. T (stripe init)", xlabel=L"T",
    ylabel=L"D_M^\parallel"
)
generate_plot!(ax2, :T, :ηk_corr_M, [:er], df; line=true) do ηk
    return real(ηk[1,1] + ηk[2,2])
end
Legend(fig[1,3], ax1, merge=true)
fig

In [ ]:
CairoMakie.activate!()

fig = Figure(size=(800, 400))
df = subset(results.data, :init_type => x -> x .== "afm_afe")
ax1 = fig[1,1] = Axis(fig,
    title="S correlation 3K/4 vs. T (afm-afe init)", xlabel=L"T",
    ylabel=L"C_{M/2}"
)
generate_plot!(ax1, :T, :sk_corr_part_K, [:er], df; line=true)
fig[1,2] = ax2 = Axis(fig,
    title="η correlation M' vs. T (afm-afe init)", xlabel=L"T",
    ylabel=L"D_M^\parallel"
)
generate_plot!(ax2, :T, :ηk_corr_M2, [:er], df; line=true) do ηk
    return real(ηk[1,1] + ηk[2,2])
end
Legend(fig[1,3], ax1, merge=true)
fig

In [ ]:
CairoMakie.activate!()

fig = Figure(size=(800, 400))
df = subset(results.data, :init_type => x -> x .== "stripe")
ax1 = fig[1,1] = Axis(fig,
    title="Energy vs. T (stripe init)", xlabel=L"T",
    ylabel=L"H"
)
generate_plot!(ax1, :T, :Energy, [:er], df; line=true)
fig[1,2] = ax2 = Axis(fig,
    title="Heat Capacity vs. T (stripe init)", xlabel=L"T",
    ylabel=L"χ"
)
generate_plot!(ax2, :T, :HeatCap, [:er], df; line=true)
Legend(fig[1,3], ax1, merge=true)
fig

In [ ]:
CairoMakie.activate!()

fig = Figure(size=(800, 400))
df = subset(results.data, :init_type => x -> x .== "afm_afe")
ax1 = fig[1,1] = Axis(fig,
    title="Energy vs. T (afm-afe init)", xlabel=L"T",
    ylabel=L"H"
)
generate_plot!(ax1, :T, :Energy, [:er], df; line=true)
fig[1,2] = ax2 = Axis(fig,
    title="Heat Capacity vs. T (afm-afe init)", xlabel=L"T",
    ylabel=L"χ"
)
generate_plot!(ax2, :T, :HeatCap, [:er], df; line=true)
Legend(fig[1,3], ax1, merge=true)
fig

In [ ]:
generate_spinks("stripe-anneal-am4", 3+15*6)

In [ ]:
fig = Figure(size=(800, 400))

fig[1,1] = ax = Axis(fig, title=L"In-plane $\eta$ M correlation vs. T", xlabel="T", ylabel="ηk")
generate_plot!(ax, :T, [:ηk_corr_M, :ηk_corr_M2, :ηk_corr_M3], [:er], results.data; line=true) do ηk1, ηk2, ηk3
    ηk = ηk1 + ηk2 + ηk3
    real(ηk[1,1] + ηk[2,2])
end
fig[1,2] = ax = Axis(fig, title=L"\text{Sk 3K/4 (C3 invariant) correlation vs. T}", xlabel="T", ylabel="ηk")
generate_plot!(ax, :T, [:sk_corr_part_K, :sk_corr_part_K2, :sk_corr_part_K3], [:er], results.data; line=true) do sk1, sk2, sk3
    sk1 + sk2 + sk3
end
Legend(fig[1,3], ax, merge=true)
fig

In [ ]:
mctimes = get_mctime_data(results, :Energy, :sk_corr_half_M, :sk_corr_part_K3, :ηk_corr_M)
nothing

In [ ]:
CairoMakie.activate!()
i = 3

var1 = :sk_corr_part_K3
var2 = :ηk_corr_M
fig = Figure(size=(800, 400))
fig[1,1] = ax1 = Axis(fig, title="$var1 vs. Bin #", xlabel="Bin #", ylabel="$var1")
fig[1,2] = ax2 = Axis(fig, title="$var2 vs. Bin #", xlabel="Bin #", ylabel="$var2")
for j in 4:6
    lines!(ax1, real.(mctimes[i + 15j][:, var1]))
    lines!(ax2, [real(ηk[1,1] + ηk[2,2]) for ηk in mctimes[i + 15j][:, var2]])
end
fig

## am = 6

In [ ]:
results = JobResult("../jobs", "am6")

In [ ]:
CairoMakie.activate!()

fig = Figure(size=(800, 400))
df = subset(results.data, :init_type => x -> x .== "stripe")
ax1 = fig[1,1] = Axis(fig,
    title="S correlation half M vs. T (stripe init)", xlabel=L"T",
    ylabel=L"C_{M/2}"
)
generate_plot!(ax1, :T, :sk_corr_half_M, [:er], df; line=true)
fig[1,2] = ax2 = Axis(fig,
    title="η correlation M vs. T (stripe init)", xlabel=L"T",
    ylabel=L"D_M^\parallel"
)
generate_plot!(ax2, :T, :ηk_corr_M, [:er], df; line=true) do ηk
    return real(ηk[1,1] + ηk[2,2])
end
Legend(fig[1,3], ax1, merge=true)
fig

In [ ]:
CairoMakie.activate!()

fig = Figure(size=(800, 400))
df = subset(results.data, :init_type => x -> x .== "afm_afe")
ax1 = fig[1,1] = Axis(fig,
    title="S correlation 3K/4 vs. T (afm-afe init)", xlabel=L"T",
    ylabel=L"C_{M/2}"
)
generate_plot!(ax1, :T, :sk_corr_part_K, [:er], df; line=true)
fig[1,2] = ax2 = Axis(fig,
    title="η correlation M' vs. T (afm-afe init)", xlabel=L"T",
    ylabel=L"D_M^\parallel"
)
generate_plot!(ax2, :T, :ηk_corr_M2, [:er], df; line=true) do ηk
    return real(ηk[1,1] + ηk[2,2])
end
Legend(fig[1,3], ax1, merge=true)
fig

In [ ]:
generate_spinks("am6", 13)

In [ ]:
mctimes = get_mctime_data(results, :Energy, :sk_corr_half_M, :sk_corr_part_K3, :ηk_corr_M)
nothing

In [ ]:
CairoMakie.activate!()
i = 3

var1 = :sk_corr_half_M
var2 = :ηk_corr_M
fig = Figure(size=(800, 400))
fig[1,1] = ax1 = Axis(fig, title="$var1 vs. Bin #", xlabel="Bin #", ylabel="$var1")
fig[1,2] = ax2 = Axis(fig, title="$var2 vs. Bin #", xlabel="Bin #", ylabel="$var2")
for j in 0:6
    lines!(ax1, real.(mctimes[i + 15j][:, var1]))
    lines!(ax2, [real(ηk[1,1] + ηk[2,2]) for ηk in mctimes[i + 15j][:, var2]])
end
fig

## am = 8

In [ ]:
results = JobResult("../jobs", "am8-low")

In [ ]:
CairoMakie.activate!()

fig = Figure(size=(800, 400))
df = subset(results.data, :init_type => x -> x .== "stripe")
ax1 = fig[1,1] = Axis(fig,
    title="S correlation half M vs. T (stripe init)", xlabel=L"T",
)
generate_plot!(ax1, :T, :sk_corr_half_M, [:er], df; line=true)
fig[1,2] = ax2 = Axis(fig,
    title="η correlation M vs. T (stripe init)", xlabel=L"T",
)
generate_plot!(ax2, :T, :ηk_corr_M, [:er], df; line=true) do ηk
    return real(ηk[1,1] + ηk[2,2])
end
Legend(fig[1,3], ax1, merge=true)
fig

In [ ]:
CairoMakie.activate!()

fig = Figure(size=(800, 400))
df = subset(results.data, :init_type => x -> x .== "stripe")
ax1 = fig[1,1] = Axis(fig,
    title="S correlation 3K/4'' vs. T (stripe init)", xlabel=L"T",
)
generate_plot!(ax1, :T, :sk_corr_part_K3, [:er], df; line=true)
fig[1,2] = ax2 = Axis(fig,
    title="η correlation M vs. T (stripe init)", xlabel=L"T",
)
generate_plot!(ax2, :T, :ηk_corr_M, [:er], df; line=true) do ηk
    return real(ηk[1,1] + ηk[2,2])
end
Legend(fig[1,3], ax1, merge=true)
fig

In [ ]:
CairoMakie.activate!()

fig = Figure(size=(800, 400))
df = subset(results.data, :init_type => x -> x .== "afm_fe")
ax1 = fig[1,1] = Axis(fig,
    title="S correlation M vs. T (AFM-FE init)", xlabel=L"T",
)
generate_plot!(ax1, :T, :sk_corr_M, [:er], df; line=true)
fig[1,2] = ax2 = Axis(fig,
    title="η correlation Γ vs. T (AFM-FE init)", xlabel=L"T",
)
generate_plot!(ax2, :T, :ηk_corr_Γ, [:er], df; line=true) do ηk
    return real(ηk[1,1] + ηk[2,2])
end
Legend(fig[1,3], ax1, merge=true)
fig

In [ ]:
CairoMakie.activate!()

fig = Figure(size=(800, 400))
df = subset(results.data, :init_type => x -> x .== "afm_afe")
ax1 = fig[1,1] = Axis(fig,
    title="S correlation 3K/4 vs. T (afm-afe init)", xlabel=L"T",
)
generate_plot!(ax1, :T, :sk_corr_part_K, [:er], df; line=true)
fig[1,2] = ax2 = Axis(fig,
    title="η correlation M' vs. T (afm-afe init)", xlabel=L"T",
)
generate_plot!(ax2, :T, :ηk_corr_M2, [:er], df; line=true) do ηk
    return real(ηk[1,1] + ηk[2,2])
end
Legend(fig[1,3], ax1, merge=true)
fig

In [ ]:
generate_spinks("am8-low", 34)

In [ ]:
mctimes = get_mctime_data(results, :Energy, :sk_corr_half_M, :sk_corr_part_K3, :ηk_corr_M, :ηk_corr_M2)
nothing

In [ ]:
CairoMakie.activate!()
i = 31

var1 = :sk_corr_half_M
var2 = :ηk_corr_M
fig = Figure(size=(800, 400))
fig[1,1] = ax1 = Axis(fig, title="$var1 vs. Bin #", xlabel="Bin #", ylabel="$var1")
fig[1,2] = ax2 = Axis(fig, title="$var2 vs. Bin #", xlabel="Bin #", ylabel="$var2")
for j in 0:0
    lines!(ax1, real.(mctimes[i + 15j][:, var1]))
    lines!(ax2, [real(ηk[1,1] + ηk[2,2]) for ηk in mctimes[i + 15j][:, var2]])
end
fig

## am = 9

In [ ]:
results = JobResult("../jobs", "am9")

In [ ]:
CairoMakie.activate!()

fig = Figure(size=(800, 400))
df = subset(results.data, :init_type => x -> x .== "fm")
ax1 = fig[1,1] = Axis(fig,
    title="S correlation Γ vs. T (fm init)", xlabel=L"T",
    ylabel=L"C_{M/2}"
)
generate_plot!(ax1, :T, :sk_corr_Γ, [:er], df; line=true)
fig[1,2] = ax2 = Axis(fig,
    title="ηz vs. T (fm init)", xlabel=L"T",
    ylabel=L"D_M^\parallel"
)
generate_plot!(ax2, :T, :ηz, [:er], df; line=true)
Legend(fig[1,3], ax1, merge=true)
fig

In [ ]:
CairoMakie.activate!()

fig = Figure(size=(800, 400))
df = subset(results.data, :init_type => x -> x .== "afm_fe")
ax1 = fig[1,1] = Axis(fig,
    title="S correlation M vs. T (afm-fe init)", xlabel=L"T",
    ylabel=L"C_{M/2}"
)
generate_plot!(ax1, :T, :sk_corr_M, [:er], df; line=true)
fig[1,2] = ax2 = Axis(fig,
    title="η correlation M' vs. T (afm-fe init)", xlabel=L"T",
    ylabel=L"D_M^\parallel"
)
generate_plot!(ax2, :T, :ηk_corr_Γ, [:er], df; line=true) do ηk
    return real(ηk[1,1] + ηk[2,2])
end
Legend(fig[1,3], ax1, merge=true)
fig

In [ ]:
generate_spinks("am9", 1)

In [ ]:
mctimes = get_mctime_data(results, :Energy, :sk_corr_M, :sk_corr_Γ, :ηk_corr_Γ, :ηz)
groupsize = 10
groups = 7
nothing

In [ ]:
CairoMakie.activate!()
i = 3

var1 = :sk_corr_M
var2 = :ηk_corr_Γ
fig = Figure(size=(800, 400))
fig[1,1] = ax1 = Axis(fig, title="$var1 vs. Bin #", xlabel="Bin #", ylabel="$var1")
fig[1,2] = ax2 = Axis(fig, title="$var2 vs. Bin #", xlabel="Bin #", ylabel="$var2")
for j in 0:groups-1
    df = mctimes[i + groupsize*j]
    er = results.data[i+groupsize*j, :er]
    lines!(ax1, df[:, var1] .- results.data[i+groupsize*j, var1], label="er=$er")
    lines!(ax2, getetapar.(df[:, var2]) .- getetapar(results.data[i+groupsize*j, var2]))
end
Legend(fig[1,3], ax1, merge=true)
fig

In [ ]:
CairoMakie.activate!()
i = 3

var1 = :sk_corr_Γ
var2 = :ηz
fig = Figure(size=(800, 400))
fig[1,1] = ax1 = Axis(fig, title="$var1 vs. Bin #", xlabel="Bin #", ylabel="$var1")
fig[1,2] = ax2 = Axis(fig, title="$var2 vs. Bin #", xlabel="Bin #", ylabel="$var2")
for j in 0:groups-1
    df = mctimes[i + groupsize*j]
    er = results.data[i+groupsize*j, :er]
    lines!(ax1, df[:, var1] .- results.data[i+groupsize*j, var1], label="er=$er")
    lines!(ax2, df[:, var2] .- results.data[i+groupsize*j, var2], label="er=$er")
end
Legend(fig[1,3], ax1, merge=true)
fig

## er = 7

In [ ]:
results = JobResult("../jobs", "er7")

In [ ]:
fig = Figure(size=(1000, 600))

setL = 24
setT = 0.5
ax1 = fig[1,1] = Axis(fig, title=L"$\eta$ \textbf{order vs} $a_m$ (T=%$setT)", xlabel=L"a_m")
df = subset(results.data, :T => T -> T .== setT, :Lx => L -> L .== setL)
generate_plot!(ax1, :am, :etak_corr_stripe, df; line=true, label="stripe")
generate_plot!(ax1, :am, :etak_corr_afm_fe, df; line=true, label="afm-fe")
generate_plot!(ax1, :am, :etak_corr_afm_afe, df; line=true, label="afm-afe")

setT = 1.0
ax2 = fig[2,1] = Axis(fig, title=L"$\eta$ \textbf{order vs} $a_m$ (T=%$setT)", xlabel=L"a_m")
df = subset(results.data, :T => T -> T .== setT, :Lx => L -> L .== setL)
generate_plot!(ax2, :am, :etak_corr_stripe, df; line=true)
generate_plot!(ax2, :am, :etak_corr_afm_fe, df; line=true)
generate_plot!(ax2, :am, :etak_corr_afm_afe, df; line=true)

setT = 1.5
ax1 = fig[1,2] = Axis(fig, title=L"$\eta$ \textbf{order vs} $a_m$ (T=%$setT)", xlabel=L"a_m")
df = subset(results.data, :T => T -> T .== setT, :Lx => L -> L .== setL)
generate_plot!(ax1, :am, :etak_corr_stripe, df; line=true, label="stripe")
generate_plot!(ax1, :am, :etak_corr_afm_fe, df; line=true, label="afm-fe")
generate_plot!(ax1, :am, :etak_corr_afm_afe, df; line=true, label="afm-afe")

setT = 2.0
ax2 = fig[2,2] = Axis(fig, title=L"$\eta$ \textbf{order vs} $a_m$ (T=%$setT)", xlabel=L"a_m")
df = subset(results.data, :T => T -> T .== setT, :Lx => L -> L .== setL)
generate_plot!(ax2, :am, :etak_corr_stripe, df; line=true)
generate_plot!(ax2, :am, :etak_corr_afm_fe, df; line=true)
generate_plot!(ax2, :am, :etak_corr_afm_afe, df; line=true)

setT = 3.0
ax1 = fig[1,3] = Axis(fig, title=L"$\eta$ \textbf{order vs} $a_m$ (T=%$setT)", xlabel=L"a_m")
df = subset(results.data, :T => T -> T .== setT, :Lx => L -> L .== setL)
generate_plot!(ax1, :am, :etak_corr_stripe, df; line=true, label="stripe")
generate_plot!(ax1, :am, :etak_corr_afm_fe, df; line=true, label="afm-fe")
generate_plot!(ax1, :am, :etak_corr_afm_afe, df; line=true, label="afm-afe")

setT = 4.0
ax2 = fig[2,3] = Axis(fig, title=L"$\eta$ \textbf{order vs} $a_m$ (T=%$setT)", xlabel=L"a_m")
df = subset(results.data, :T => T -> T .== setT, :Lx => L -> L .== setL)
generate_plot!(ax2, :am, :etak_corr_stripe, df; line=true)
generate_plot!(ax2, :am, :etak_corr_afm_fe, df; line=true)
generate_plot!(ax2, :am, :etak_corr_afm_afe, df; line=true)

Legend(fig[:,4], ax1, merge=true)
fig

In [ ]:
fig = Figure(size=(1000, 600))

setL = 48
setT = 0.5
ax1 = fig[1,1] = Axis(fig, title=L"$\eta$ \textbf{order vs} $a_m$ (T=%$setT)", xlabel=L"a_m")
df = subset(results.data, :T => T -> T .== setT, :Lx => L -> L .== setL)
generate_plot!(ax1, :am, :etak_corr_stripe, df; line=true, label="stripe")
generate_plot!(ax1, :am, :etak_corr_afm_fe, df; line=true, label="afm-fe")
generate_plot!(ax1, :am, :etak_corr_afm_afe, df; line=true, label="afm-afe")

setT = 1.0
ax2 = fig[2,1] = Axis(fig, title=L"$\eta$ \textbf{order vs} $a_m$ (T=%$setT)", xlabel=L"a_m")
df = subset(results.data, :T => T -> T .== setT, :Lx => L -> L .== setL)
generate_plot!(ax2, :am, :etak_corr_stripe, df; line=true)
generate_plot!(ax2, :am, :etak_corr_afm_fe, df; line=true)
generate_plot!(ax2, :am, :etak_corr_afm_afe, df; line=true)

setT = 1.5
ax1 = fig[1,2] = Axis(fig, title=L"$\eta$ \textbf{order vs} $a_m$ (T=%$setT)", xlabel=L"a_m")
df = subset(results.data, :T => T -> T .== setT, :Lx => L -> L .== setL)
generate_plot!(ax1, :am, :etak_corr_stripe, df; line=true, label="stripe")
generate_plot!(ax1, :am, :etak_corr_afm_fe, df; line=true, label="afm-fe")
generate_plot!(ax1, :am, :etak_corr_afm_afe, df; line=true, label="afm-afe")

setT = 2.0
ax2 = fig[2,2] = Axis(fig, title=L"$\eta$ \textbf{order vs} $a_m$ (T=%$setT)", xlabel=L"a_m")
df = subset(results.data, :T => T -> T .== setT, :Lx => L -> L .== setL)
generate_plot!(ax2, :am, :etak_corr_stripe, df; line=true)
generate_plot!(ax2, :am, :etak_corr_afm_fe, df; line=true)
generate_plot!(ax2, :am, :etak_corr_afm_afe, df; line=true)

setT = 3.0
ax1 = fig[1,3] = Axis(fig, title=L"$\eta$ \textbf{order vs} $a_m$ (T=%$setT)", xlabel=L"a_m")
df = subset(results.data, :T => T -> T .== setT, :Lx => L -> L .== setL)
generate_plot!(ax1, :am, :etak_corr_stripe, df; line=true, label="stripe")
generate_plot!(ax1, :am, :etak_corr_afm_fe, df; line=true, label="afm-fe")
generate_plot!(ax1, :am, :etak_corr_afm_afe, df; line=true, label="afm-afe")

setT = 4.0
ax2 = fig[2,3] = Axis(fig, title=L"$\eta$ \textbf{order vs} $a_m$ (T=%$setT)", xlabel=L"a_m")
df = subset(results.data, :T => T -> T .== setT, :Lx => L -> L .== setL)
generate_plot!(ax2, :am, :etak_corr_stripe, df; line=true)
generate_plot!(ax2, :am, :etak_corr_afm_fe, df; line=true)
generate_plot!(ax2, :am, :etak_corr_afm_afe, df; line=true)

Legend(fig[:,4], ax1, merge=true)
fig

In [ ]:
fig = Figure(size=(1000, 600))

setL = 48
setT = 0.5
ax1 = fig[1,1] = Axis(fig, title=L"$\eta$ \textbf{order vs} $a_m$ (T=%$setT)", xlabel=L"a_m")
df = subset(results.data, :T => T -> T .== setT, :Lx => L -> L .== setL)
generate_plot!(ax1, :am, :sk_corr_half_M, df; line=true, label="stripe")
generate_plot!(ax1, :am, :sk_corr_M, df; line=true, label="afm-fe")
generate_plot!(ax1, :am, :sk_corr_part_K, df; line=true, label="afm-afe")

setT = 1.0
ax2 = fig[2,1] = Axis(fig, title=L"$\eta$ \textbf{order vs} $a_m$ (T=%$setT)", xlabel=L"a_m")
df = subset(results.data, :T => T -> T .== setT, :Lx => L -> L .== setL)
generate_plot!(ax2, :am, :sk_corr_half_M, df; line=true, label="stripe")
generate_plot!(ax2, :am, :sk_corr_M, df; line=true, label="afm-fe")
generate_plot!(ax2, :am, :sk_corr_part_K, df; line=true, label="afm-afe")

setT = 1.5
ax1 = fig[1,2] = Axis(fig, title=L"$\eta$ \textbf{order vs} $a_m$ (T=%$setT)", xlabel=L"a_m")
df = subset(results.data, :T => T -> T .== setT, :Lx => L -> L .== setL)
generate_plot!(ax1, :am, :sk_corr_half_M, df; line=true, label="stripe")
generate_plot!(ax1, :am, :sk_corr_M, df; line=true, label="afm-fe")
generate_plot!(ax1, :am, :sk_corr_part_K, df; line=true, label="afm-afe")

setT = 2.0
ax2 = fig[2,2] = Axis(fig, title=L"$\eta$ \textbf{order vs} $a_m$ (T=%$setT)", xlabel=L"a_m")
df = subset(results.data, :T => T -> T .== setT, :Lx => L -> L .== setL)
generate_plot!(ax2, :am, :sk_corr_half_M, df; line=true, label="stripe")
generate_plot!(ax2, :am, :sk_corr_M, df; line=true, label="afm-fe")
generate_plot!(ax2, :am, :sk_corr_part_K, df; line=true, label="afm-afe")

setT = 3.0
ax1 = fig[1,3] = Axis(fig, title=L"$\eta$ \textbf{order vs} $a_m$ (T=%$setT)", xlabel=L"a_m")
df = subset(results.data, :T => T -> T .== setT, :Lx => L -> L .== setL)
generate_plot!(ax1, :am, :sk_corr_half_M, df; line=true, label="stripe")
generate_plot!(ax1, :am, :sk_corr_M, df; line=true, label="afm-fe")
generate_plot!(ax1, :am, :sk_corr_part_K, df; line=true, label="afm-afe")

setT = 4.0
ax2 = fig[2,3] = Axis(fig, title=L"$\eta$ \textbf{order vs} $a_m$ (T=%$setT)", xlabel=L"a_m")
df = subset(results.data, :T => T -> T .== setT, :Lx => L -> L .== setL)
generate_plot!(ax2, :am, :sk_corr_half_M, df; line=true, label="stripe")
generate_plot!(ax2, :am, :sk_corr_M, df; line=true, label="afm-fe")
generate_plot!(ax2, :am, :sk_corr_part_K, df; line=true, label="afm-afe")

Legend(fig[:,4], ax1, merge=true)
fig

In [ ]:
fig = Figure(size=(600, 600))

ax1 = fig[1,1] = Axis(fig, title=L"$\eta$ \textbf{stripe kurtosis vs} $a_m$", xlabel=L"a_m")
df = subset(results.data, :Lx => L -> L .== 24)
generate_plot!(ax1, :am, :etak_kurt_stripe, [:T], df; line=true)

ax2 = fig[1,2] = Axis(fig, title=L"$\eta$ \textbf{afm-fe kurtosis vs} $a_m$", xlabel=L"a_m")
generate_plot!(ax2, :am, :etak_kurt_afm_fe, [:T], df; line=true)

ax3 = fig[2,1] = Axis(fig, title=L"$\eta$ \textbf{afm-afe kurtosis vs} $a_m$", xlabel=L"a_m")
generate_plot!(ax3, :am, :etak_kurt_afm_afe, [:T], df; line=true)

Legend(fig[2,2], ax1, merge=true, tellwidth=false)
fig

In [ ]:
fig = Figure(size=(600, 600))

ax1 = fig[1,1] = Axis(fig, title=L"$\eta$ \textbf{stripe kurtosis vs} $a_m$", xlabel=L"a_m")
df = subset(results.data, :Lx => L -> L .== 48)
generate_plot!(ax1, :am, :etak_kurt_stripe, [:T], df; line=true)

ax2 = fig[1,2] = Axis(fig, title=L"$\eta$ \textbf{afm-fe kurtosis vs} $a_m$", xlabel=L"a_m")
generate_plot!(ax2, :am, :etak_kurt_afm_fe, [:T], df; line=true)

ax3 = fig[2,1] = Axis(fig, title=L"$\eta$ \textbf{afm-afe kurtosis vs} $a_m$", xlabel=L"a_m")
generate_plot!(ax3, :am, :etak_kurt_afm_afe, [:T], df; line=true)

Legend(fig[2,2], ax1, merge=true, tellwidth=false)
fig

In [ ]:
fig = Figure(size=(800, 400))

df = results.data
ax1 = fig[1,1] = Axis(fig, title="Energy vs. am", xlabel=L"a_m")
generate_plot!(ax1, :am, :Energy, [:T], df; line=true)
ax2 = fig[1,2] = Axis(fig, title="Heat Capacity vs. am", xlabel=L"a_m")
generate_plot!(ax2, :am, :HeatCap, [:T], df; line=true)

Legend(fig[1,3], ax1, merge=true)
fig

In [ ]:
mctimes = get_mctime_data(results, :Energy, :sk_corr_M, :sk_corr_half_M, :sk_corr_part_K, :ηk_corr_Γ, :ηk_corr_M, :ηk_corr_M2, :ηz)
groups = 3
groupsize = 32
nothing

In [ ]:
fig = Figure(size=(1200,800))
i = 11

scorrs = [:sk_corr_half_M, :sk_corr_M, :sk_corr_part_K]
ηcorrs = [:ηk_corr_M, :ηk_corr_Γ, :ηk_corr_M2]
for j in 1:groups
    fig[j,1] = ax1 = Axis(fig, title="$(scorrs[j])")
    fig[j,2] = ax2 = Axis(fig, title="$(ηcorrs[j])")
    df = mctimes[(j-1)*groupsize + i]
    lines!(ax1, df[:, scorrs[j]])
    lines!(ax2, getetapar.(df[:, ηcorrs[j]]))
end
println("(T, am) = ($(results.data[i,:T]), $(results.data[i,:am]))")
fig

In [ ]:
fig = Figure(size=(1200,800))
i = 9

scorrs = [:Energy, :Energy, :Energy]
ηcorrs = [:Energy, :Energy, :Energy]
for j in 1:groups
    fig[j,1] = ax1 = Axis(fig, title="$(scorrs[j])")
    fig[j,2] = ax2 = Axis(fig, title="$(ηcorrs[j])")
    df = mctimes[(j-1)*groupsize + i]
    lines!(ax1, df[:, scorrs[j]])
    lines!(ax2, df[:, ηcorrs[j]])
end
println("(T, am) = ($(results.data[i,:T]), $(results.data[i,:am]))")
fig

## Full Phase Diagram

In [ ]:
ams = 4:11
ers = 5:11
Ts = [0.5, 0.75, 2.0, 4.0]
results = JobResult("../jobs", "full-diagram")

In [ ]:
function getscorr(df, setam, seter, phase)
    df = subset(df, :am => am -> am .== setam, :er => er -> er .== seter)
    if phase == :stripe
        return sum(df[1, [:sk_corr_near_half_M, :sk_corr_near_half_M2, :sk_corr_near_half_M3]])
    elseif phase == :fm
        return df[1, :sk_corr_near_Γ]
    elseif phase == :afm_fe
        return sum(df[1, [:sk_corr_near_M, :sk_corr_near_M2, :sk_corr_near_M3]])
    elseif phase == :afm_afe
        return sum(df[1, [:sk_corr_part_K, :sk_corr_part_K2, :sk_corr_part_K3]])
    end
end
function getηcorr(df, setam, seter, phase)
    df = subset(df, :am => am -> am .== setam, :er => er -> er .== seter)
    a1 = [1 0 0]
    a2 = [-1/2 √3/2 0]
    a3 = [-1/2 -√3/2 0]
    y1 = [0 1 0]
    y2 = [-√3/2 -1/2 0]
    y3 = [√3/2 -1/2 0]
    if phase == :stripe
        corr = a3 * df[1, :ηk_corr_near_M] * a3' + a1 * df[1, :ηk_corr_near_M2] * a1' + a2 * df[1, :ηk_corr_near_M3] * a2'
        return real(first(corr))
    elseif phase == :fm
        return real(df[1, :ηk_corr_near_Γ][3,3])
    elseif phase == :afm_fe
        return real(getetapar(df[1, :ηk_corr_near_Γ]))
    elseif phase == :afm_afe
        corr = y1 * df[1, :ηk_corr_M] * y1' + y2 * df[1, :ηk_corr_M2] * y2' + y3 * df[1, :ηk_corr_M3] * y3'
        return real(first(corr))
    end
end

In [ ]:
for phase in [:stripe, :fm, :afm_fe, :afm_afe]
    fig = Figure(size=(800,800))
    fig[1,1] = ax11 = Axis(fig, title=L"\textbf{Stripe Spin Order} $(T=%$setT)$", xlabel = L"a_m", ylabel=L"\epsilon_r")
    fig[1,3] = ax12 = Axis(fig, title=L"\textbf{FM Spin Order} $(T=%$setT)$", xlabel = L"a_m", ylabel=L"\epsilon_r")
    fig[2,1] = ax21 = Axis(fig, title=L"\textbf{AFM-FE Spin Order} $(T=%$setT)$", xlabel = L"a_m", ylabel=L"\epsilon_r")
    fig[2,3] = ax22 = Axis(fig, title=L"\textbf{AFM-AFE Spin Order} $(T=%$setT)$", xlabel = L"a_m", ylabel=L"\epsilon_r")

    df = subset(results.data, :T => T -> T .== 0.5)
    corrs = reshape([2*getscorr(df,am,er,phase).val for (am,er) in Iterators.product(ams,ers)], (length(ams), length(ers)))
    hm = heatmap!(ax11, ams, ers, corrs)
    Colorbar(fig[1,2], hm)
    df = subset(results.data, :T => T -> T .== 0.75)
    corrs = reshape([getscorr(df,am,er,phase).val for (am,er) in Iterators.product(ams,ers)], (length(ams), length(ers)))
    hm = heatmap!(ax12, ams, ers, corrs)
    Colorbar(fig[1,4], hm)
    df = subset(results.data, :T => T -> T .== 2.0)
    corrs = reshape([2*getscorr(df,am,er,phase).val for (am,er) in Iterators.product(ams,ers)], (length(ams), length(ers)))
    hm = heatmap!(ax21, ams, ers, corrs)
    Colorbar(fig[2,2], hm)
    df = subset(results.data, :T => T -> T .== 4.0)
    corrs = reshape([2*getscorr(df,am,er,phase).val for (am,er) in Iterators.product(ams,ers)], (length(ams), length(ers)))
    hm = heatmap!(ax22, ams, ers, corrs)
    Colorbar(fig[2,4], hm)

    save("plots/full_diagram_spin_$phase.png", fig)
end

In [ ]:
for phase in [:stripe, :fm, :afm_fe, :afm_afe]
    fig = Figure(size=(800,800))
    fig[1,1] = ax11 = Axis(fig, title=L"\textbf{Stripe Spin Order} $(T=%$setT)$", xlabel = L"a_m", ylabel=L"\epsilon_r")
    fig[1,3] = ax12 = Axis(fig, title=L"\textbf{FM Spin Order} $(T=%$setT)$", xlabel = L"a_m", ylabel=L"\epsilon_r")
    fig[2,1] = ax21 = Axis(fig, title=L"\textbf{AFM-FE Spin Order} $(T=%$setT)$", xlabel = L"a_m", ylabel=L"\epsilon_r")
    fig[2,3] = ax22 = Axis(fig, title=L"\textbf{AFM-AFE Spin Order} $(T=%$setT)$", xlabel = L"a_m", ylabel=L"\epsilon_r")

    df = subset(results.data, :T => T -> T .== 0.5)
    corrs = reshape([2*getscorr(df,am,er,phase).val for (am,er) in Iterators.product(ams,ers)], (length(ams), length(ers)))
    hm = heatmap!(ax11, ams, ers, corrs)
    Colorbar(fig[1,2], hm)
    df = subset(results.data, :T => T -> T .== 0.75)
    corrs = reshape([getscorr(df,am,er,phase).val for (am,er) in Iterators.product(ams,ers)], (length(ams), length(ers)))
    hm = heatmap!(ax12, ams, ers, corrs)
    Colorbar(fig[1,4], hm)
    df = subset(results.data, :T => T -> T .== 2.0)
    corrs = reshape([2*getscorr(df,am,er,phase).val for (am,er) in Iterators.product(ams,ers)], (length(ams), length(ers)))
    hm = heatmap!(ax21, ams, ers, corrs)
    Colorbar(fig[2,2], hm)
    df = subset(results.data, :T => T -> T .== 4.0)
    corrs = reshape([2*getscorr(df,am,er,phase).val for (am,er) in Iterators.product(ams,ers)], (length(ams), length(ers)))
    hm = heatmap!(ax22, ams, ers, corrs)
    Colorbar(fig[2,4], hm)

    save("plots/full_diagram_spin_err_$phase.png", fig)
end

In [ ]:
for phase in [:stripe, :fm, :afm_fe, :afm_afe]
    fig = Figure(size=(800,800))
    name = replace
    fig[1,1] = ax11 = Axis(fig, title=L"\textbf{%$(replace(String(phase), '_' => '-')) Eta Order} $(T=0.5)$", xlabel = L"a_m", ylabel=L"\epsilon_r")
    fig[1,3] = ax12 = Axis(fig, title=L"\textbf{%$(replace(String(phase), '_' => '-')) Eta Order} $(T=0.75)$", xlabel = L"a_m", ylabel=L"\epsilon_r")
    fig[2,1] = ax21 = Axis(fig, title=L"\textbf{%$(replace(String(phase), '_' => '-')) Eta Order} $(T=2.0)$", xlabel = L"a_m", ylabel=L"\epsilon_r")
    fig[2,3] = ax22 = Axis(fig, title=L"\textbf{%$(replace(String(phase), '_' => '-')) Eta Order} $(T=4.0)$", xlabel = L"a_m", ylabel=L"\epsilon_r")

    df = subset(results.data, :T => T -> T .== 0.5)
    corrs = reshape([getηcorr(df,am,er,phase).val for (am,er) in Iterators.product(ams,ers)], (length(ams), length(ers)))
    hm = heatmap!(ax11, ams, ers, corrs)
    Colorbar(fig[1,2], hm)
    df = subset(results.data, :T => T -> T .== 0.75)
    corrs = reshape([getηcorr(df,am,er,phase).val for (am,er) in Iterators.product(ams,ers)], (length(ams), length(ers)))
    hm = heatmap!(ax12, ams, ers, corrs)
    Colorbar(fig[1,4], hm)
    df = subset(results.data, :T => T -> T .== 2.0)
    corrs = reshape([getηcorr(df,am,er,phase).val for (am,er) in Iterators.product(ams,ers)], (length(ams), length(ers)))
    hm = heatmap!(ax21, ams, ers, corrs)
    Colorbar(fig[2,2], hm)
    df = subset(results.data, :T => T -> T .== 4.0)
    corrs = reshape([getηcorr(df,am,er,phase).val for (am,er) in Iterators.product(ams,ers)], (length(ams), length(ers)))
    hm = heatmap!(ax22, ams, ers, corrs)
    Colorbar(fig[2,4], hm)

    save("plots/full_diagram_eta_$phase.png", fig)
end

In [ ]:
for setT in Ts
    fig = Figure(size=(800,800))
    fig[1,1] = ax11 = Axis(fig, title=L"\textbf{Stripe Eta Stddev} $(T=%$setT)$", xlabel = L"a_m", ylabel=L"\epsilon_r")
    fig[1,3] = ax12 = Axis(fig, title=L"\textbf{FM Eta Stddev} $(T=%$setT)$", xlabel = L"a_m", ylabel=L"\epsilon_r")
    fig[2,1] = ax21 = Axis(fig, title=L"\textbf{AFM-FE Eta Stddev} $(T=%$setT)$", xlabel = L"a_m", ylabel=L"\epsilon_r")
    fig[2,3] = ax22 = Axis(fig, title=L"\textbf{AFM-AFE Eta Stddev} $(T=%$setT)$", xlabel = L"a_m", ylabel=L"\epsilon_r")

    maxcorr = 2
    df = subset(results.data, :T => T -> T .== setT)
    corrs = reshape([2*getηcorr(df,am,er,:stripe).err for (am,er) in Iterators.product(ams,ers)], (length(ams), length(ers)))
    hm = heatmap!(ax11, ams, ers, corrs)
    Colorbar(fig[1,2], hm)
    corrs = reshape([getηcorr(df,am,er,:fm).err for (am,er) in Iterators.product(ams,ers)], (length(ams), length(ers)))
    hm = heatmap!(ax12, ams, ers, corrs)
    Colorbar(fig[1,4], hm)
    corrs = reshape([2*getηcorr(df,am,er,:afm_fe).err for (am,er) in Iterators.product(ams,ers)], (length(ams), length(ers)))
    hm = heatmap!(ax21, ams, ers, corrs)
    Colorbar(fig[2,2], hm)
    corrs = reshape([2*getηcorr(df,am,er,:afm_afe).err for (am,er) in Iterators.product(ams,ers)], (length(ams), length(ers)))
    hm = heatmap!(ax22, ams, ers, corrs)
    Colorbar(fig[2,4], hm)

    save("plots/full_diagram_eta_errT$(setT).png", fig)
end

In [ ]:
mctimes = get_mctime_data(results, :sk_corr_near_Γ, :ηk_corr_near_Γ,
    :sk_corr_near_M, :sk_corr_near_M2, :sk_corr_near_M3,
    :sk_corr_near_half_M, :sk_corr_near_half_M2, :sk_corr_near_half_M3,
    :sk_corr_near_part_K, :sk_corr_near_part_K2, :sk_corr_near_part_K3,
)
groups = length(Ts)
groupsize = length(ams) * length(ers)
nothing

In [ ]:
i = 46
fig = Figure(size=(800, 500))
for j in 1:groups
    df = mctimes[(j-1) * groupsize + i]
    corrs = df[:, :sk_corr_near_M] + df[:, :sk_corr_near_M2] + df[:, :sk_corr_near_M3]
    params = results.data[(j-1) * groupsize + i, :]
    T = params[:T]
    am = params[:am]
    er = params[:er]
    ax = Axis(fig[fld(j-1,2)+1, mod1(j,2)], title=L"\textbf{Spin M Correlation} $(T,a_m,\epsilon_r) = (%$T,%$am,%$er)$")
    lines!(ax, corrs)
end
fig

In [ ]:
i = 12
fig = Figure(size=(800, 500))
for j in 1:groups
    df = mctimes[(j-1) * groupsize + i]
    corrs = df[:, :sk_corr_near_Γ]
    params = results.data[(j-1) * groupsize + i, :]
    T = params[:T]
    am = params[:am]
    er = params[:er]
    ax = Axis(fig[fld(j-1,2)+1, mod1(j,2)], title=L"\textbf{Spin} $\Gamma$ \textbf{Correlation} $(T,a_m,\epsilon_r) = (%$T,%$am,%$er)$")
    lines!(ax, corrs)
end
fig

In [ ]:
i = 13
fig = Figure(size=(800, 500))
for j in 1:groups
    df = mctimes[(j-1) * groupsize + i]
    corrs = df[:, :sk_corr_near_half_M] + df[:, :sk_corr_near_half_M2] + df[:, :sk_corr_near_half_M3]
    params = results.data[(j-1) * groupsize + i, :]
    T = params[:T]
    am = params[:am]
    er = params[:er]
    ax = Axis(fig[fld(j-1,2)+1, mod1(j,2)], title=L"\textbf{Spin M/2 Correlation} $(T,a_m,\epsilon_r) = (%$T,%$am,%$er)$")
    lines!(ax, corrs)
end
fig

In [ ]:
i = 21
fig = Figure(size=(800, 500))
for j in 1:groups
    df = mctimes[(j-1) * groupsize + i]
    corrs = df[:, :sk_corr_near_part_K] + df[:, :sk_corr_near_part_K2] + df[:, :sk_corr_near_part_K3]
    params = results.data[(j-1) * groupsize + i, :]
    T = params[:T]
    am = params[:am]
    er = params[:er]
    ax = Axis(fig[fld(j-1,2)+1, mod1(j,2)], title=L"\textbf{Spin 3K/4 Correlation} $(T,a_m,\epsilon_r) = (%$T,%$am,%$er)$")
    lines!(ax, corrs)
end
fig